Setup (token + connect):

In [2]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

Paste your Hugging Face READ token (hf_...): ··········


Contract (markdown cell):
give me the bullet points

One row = one content page (content_hash_id), on one day, from fact_content_daily_performance
Table used: fact_content_daily_performance, filtered to month=2026-03
Time window: March 2026 (mid-panel month)
What I'd predict: a refresh opportunity score — whether this page is declining but still has demand, so it should be reviewed
What I exclude: any outcome only known after the decision point, e.g. whether the page was actually refreshed afterward



In [7]:
# Query 1 — grain check: one row = one content_hash_id per day
con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1,2
    HAVING COUNT(*) > 1
""").df()   # should return 0 rows if grain is correct



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,n


In [8]:
# Query 2 — row count + date span
con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS start_d, MAX(report_date) AS end_d
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()



,n_rows,start_d,end_d
0,9841378,2026-03-01,2026-03-31


In [9]:
# Query 3 — availability check
con.sql(f"""
    SELECT COUNT(*) AS total,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

,total,available
0,9841378,3611061


Five features (from month=2026-03, prev-30-day style so each is knowable before the decision moment):

In [10]:
feats = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']} WHERE month = '2026-03'
    )
    SELECT f.content_hash_id,
           SUM(f.gsc_impressions) AS impressions_30d,
           SUM(f.gsc_clicks) AS clicks_30d,
           AVG(f.gsc_avg_position) AS avg_position_30d,
           SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_30d,
           COUNT(DISTINCT f.report_date) AS days_with_data
    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.month = '2026-03'
    GROUP BY 1
    HAVING impressions_30d >= 50
""").df()

feats.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions_30d,clicks_30d,avg_position_30d,ctr_30d,days_with_data
0,content_7a105f548d9c6916,6523.0,7.0,7.209549,0.001073,31
1,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.000000,31
2,content_36c36abc7650d7af,5630.0,6.0,6.724039,0.001066,31
3,content_a7da352b73b02668,4944.0,13.0,7.244844,0.002629,31
4,content_1855a661b4d36130,429.0,1.0,4.209227,0.002331,31


impressions_30d — knowable because it only sums past daily impressions, nothing from after the decision date
clicks_30d — same, only past daily clicks
avg_position_30d — average of past daily ranking positions, already observed
ctr_30d — derived only from the two past-only columns above
days_with_data — just counts how many days in the past window had any record

The trap (add a label-derived column on purpose):

In [12]:
# Fake "label" for this demo: did the page decline next period?
# (In real work this would come from a later month's data — here we simulate the trap.)
import numpy as np

feats['is_declining'] = (feats['ctr_30d'] < feats['ctr_30d'].median()).astype(int)  # placeholder label

# TRUE leak: built directly from the label itself (this is the trap)
feats['leaky_feature'] = feats['is_declining'] * 1.0 + np.random.normal(0, 0.01, len(feats))

# quick score WITH the leak
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_leak = feats[['impressions_30d','clicks_30d','avg_position_30d','ctr_30d','days_with_data','leaky_feature']]
y = feats['is_declining']
Xtr, Xte, ytr, yte = train_test_split(X_leak, y, test_size=0.25, random_state=42)
model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print('WITH leak:', accuracy_score(yte, model.predict(Xte)))

# now remove it and re-check
X_honest = feats[['impressions_30d','clicks_30d','avg_position_30d','ctr_30d','days_with_data']]
Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.25, random_state=42)
model2 = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print('WITHOUT leak:', accuracy_score(yte, model2.predict(Xte)))

WITH leak: 1.0
WITHOUT leak: 0.9972096868648592


The leaky_feature column was built directly from the label, so the model scored 1.0 — a red flag for leakage. Removing it dropped the score to 0.997, which is the honest result

A limitation of this slice is that gsc_data_available is FALSE for a large share of rows (roughly 63% in March 2026), so the feature values are built on partial daily coverage, not the full month for every page